---
title: "Cost-per-click surge attribution"
author: "Andratx Bellmunt"
abstract: >
  Adapted from a real business scenario. We use mix-rate decomposition to asses the contribution of each individual product to a global surge on cost per click. In order to get bootstrap confidence intervals on each product contribution we rely on Monte Carlo methods.
format:
  html:
    code-fold: true
    self-contained: true
    include-after-body: _tracker.html
jupyter: python3
number-sections: true
---

**Revised narrative arc**
Given the 88% mix / 11% rate split, I'd restructure the notebook slightly:

1. **Business problem:** 18% aggregate CPC increase, stakeholders are alarmed
2. **Naive approach:** look at product-level CPC changes — most products look fine, some even improved. Paradox: how can the aggregate be +18%?
3. **Simpson's Paradox:** explain with the N=2 toy example in CPC terms
4. **Shapley attempt:** natural instinct to attribute fairly — show it gives counterintuitive results and explain why (cross-sectional, not causal)
5. **Rate-mix decomposition:** the right framing. Show the 88/11 split — the problem is almost entirely compositional, not a pricing problem
6. **Attribution of the mix effect:** which products contributed most to the click share shift toward expensive inventory? This is your ranked list for stakeholders
7. **Uncertainty quantification:** bootstrap CIs on mix contributions — which attributions are robust at N=1900?

**The key message for stakeholders**
The actionable conclusion changes completely depending on which decomposition you use:

 - Naive / Shapley → "these products got more expensive, fix their CPCs"
 - Rate-mix → "click volume shifted toward expensive products, investigate why the allocation changed"

# Initialization

## Imports and settings

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

In [ ]:
# Configuration
pd.set_option("max_colwidth", None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
accent_colors = ["#009AD7", "#06C4B0", "#FF540A", "#FEC10D", "#E81159", "#002856"]

## Auxiliary functions

In [ ]:
def add_derived_columns(df: pd.DataFrame, mix_rate: bool=False) -> pd.DataFrame:
    df["cpc_before"] = df["cost_before"] / df["clicks_before"]
    df["cpc_after"] = df["cost_after"] / df["clicks_after"]
    df["cpc_delta"] = df["cpc_after"] - df["cpc_before"]

    if mix_rate:
        df["click_weight_before"] = df["clicks_before"] / df["clicks_before"].sum()
        df["click_weight_after"] = df["clicks_after"] / df["clicks_after"].sum()
        df["click_weight_delta"] = df["click_weight_after"] - df["click_weight_before"]
    else:
        df["cpc_perc_diff"] = (df["cpc_after"] / df["cpc_before"] - 1) * 100

    return df

def aggregate_data(df: pd.DataFrame, numeric_only: bool=False) -> pd.DataFrame:
    df_agg = pd.DataFrame(df.sum(numeric_only=numeric_only)).T
    df_agg["clicks_before"] = df_agg["clicks_before"].astype(int)
    df_agg["clicks_after"] = df_agg["clicks_after"].astype(int)

    return df_agg

def compute_rate_mix_effects(df: pd.DataFrame) -> pd.DataFrame:
    df["rate_effect"] = df["click_weight_after"] * df["cpc_delta"]
    df["mix_effect"] = df["click_weight_delta"] * df["cpc_before"]
    df["total_effect"] = df["rate_effect"] + df["mix_effect"]

    return df

# The data

We read the data set:
  - It contains clicks and costs data for 1,907 digital advertising campaigns
  - Data corresponds to two comparable periods, labelled "before" and "after"

In [ ]:
df_base = pd.read_csv("../assets/cpc_data.csv")
display(df_base)

We compute the cost per click (CPC) for before and after and add the delta and the percentual difference between the two:

In [ ]:
df = add_derived_columns(df_base)
display(df)

Finally, we aggregate the data to see the global effects:

In [ ]:
df_agg = add_derived_columns(aggregate_data(df, numeric_only=True))
display(df_agg)

# What the stakeholders see and do (and why it does not work)

When looking at the aggregate data the stakeholders see that global **CPC has increased +18.04%** and they are alarmed.

Due to operational constraints they cannot realistically take action on more than 100 campaigns, so they need to prioritize.

They proceed as follows:
  - Take the 100 largest campaigns in terms of clicks

  - Sort them according to percentual CPC increase
  
  - Prioritize actions according to that ranking

Let us reproduce this strategy and see what campaigns are tageted:

In [ ]:
print(
    "Top 5 prioritized campaigns according to initial stakeholder's strategy:\n",
    list(
        df
        .sort_values(by="clicks_after", ascending=False)
        .head(100)
        .sort_values(by="cpc_perc_diff", ascending=False)
        .head(5)
        ["campaign_id"]
    )
)

This strategy has two main flaws:
  - **Sorting by percentage CPC increase ignores the base CPC level.** A 20% increase on a $1 click is a +$0.20 change. A 5% increase on a $50 click is $2.50, far more impactful. Percentage change without reference to the baseline is not a measure of impact.

  - **Individual CPC increases do not necessarily imply an aggregate CPC increase.** A set of campaigns can all increase their individual CPCs while the aggregate goes down, and the reverse is equally possible. This happens when the mix of clicks across campaigns changes significantly between the two periods.

## The mix problem: Simpson's paradox

Let us illustrate with a toy example the second of the flaws that we mentioned above:

In [ ]:
df_toy_base = pd.DataFrame(
    columns=["id", "cost_before", "clicks_before", "cost_after", "clicks_after"],
    data=[
        ["A", 1000.00, 100,  200.00, 25],
        ["B",  900.00,  40,  800.00, 40]
    ]
)

df_toy = add_derived_columns(df_toy_base)

df_toy_agg = aggregate_data(df_toy)
df_toy_agg = add_derived_columns(df_toy_agg)

print("Toy example:")
display(df_toy)
print("Aggregated data:")
display(df_toy_agg)

Even in a simple example with only two campaigns we can see how the phenomenon of [Simpson's paradox](https://en.wikipedia.org/wiki/Simpson%27s_paradox) arises:

- Both campaigns **individual CPCs go down**: -20.00% for product A and -11.11% for product B

- However the **aggregated CPC goes up**: +13.36%

This example actually illustrates a very common business situation:

- Campaign A was a large cheap click campaign (CPC=$10) that lost 80% of its volume ($1,000→$200) due to e.g. severe budget cut or dried up audience. Along the way its clicks got cheaper (CPC=$8) because we do not access anymore the more expensive higher quality clicks in that segment.

- Campaign B is a smaller, more expensive campaign (CPC=$22.5) that was optimized and achieved the same number of clicks (40) with a slightly lower budget ($900→$800). This effectively reduced its cost per click (now CPC=$20)

- Both campaigns look fine individually. Yet the aggregate CPC jumps from $13.57 to $15.38 (+13.36%) purely because A's cheap clicks are no longer diluting B's expensive ones.

Hence, the main culprit is that **the mix between the two campaigns has completely changed**. We shall review this in more detail later.

Finally, note that in our real case scenario –where we have 1,900+ campaigns instead of just two– these interactions become much more complex.

# First proposed technical solution (and why it does not work)

In order to navigate the problems we exposed while keeping the stakeholders language (namely "percentual CPC changes") we can borrow a tool from game theory: [Shapley values](https://en.wikipedia.org/wiki/Shapley_value).

- Shapley values measure the contribution of each individual player to a common goal

- To us, each campaign is a player and the common goal is the aggregated CPC percentual change. We want to measure how much each individual campaign contributes to it.

- One property of Shapley values is that individual contributions always add up to the final global result (efficiency axiom). In our toy example above, the sum of the Shapley value of A and the Shapley value of B must be +13.36.

- More in general, in our real data, we shall compute the 1,900 Shapley values and all of them would add up to +18.04.

On paper, this approach works well because we can say to stakeholders "from the global +18.04%, this campaign contributed this bit". However, as we will readily see, it fails to capture the real reason behind the CPC surge.

Let us get back to our toy example and compute by the corresponding Shapley values (for only two campaigns the formula is easy to be applied directly):

In [ ]:
print("Shapley values for CPC percentual change:")
print(f"  Campaign A: {round(1/2 * (df_toy_agg['cpc_perc_diff'].iloc[0] + df_toy['cpc_perc_diff'].iloc[0] - df_toy['cpc_perc_diff'].iloc[1]), 2)}%")
print(f"  Campaign B: {round(1/2 * (df_toy_agg['cpc_perc_diff'].iloc[0] + df_toy['cpc_perc_diff'].iloc[1] - df_toy['cpc_perc_diff'].iloc[0]), 2)}%")

This points to B as the main culprit. However, we know from our construction of the example that A is the campaign whose behavior changed most dramatically (it lost 75% of its clicks). B did nothing wrong: its CPC actually decreased.

The reason Shapley misleads us here is that the marginal contribution of each campaign depends not only on its own CPC dynamics (rate effects), but on the volume composition of the coalition it enters (mix effects). A's dramatic volume loss distorts every coalition it participates in, and that distortion gets reflected in B's attributed contribution.

More fundamentally, CPC percentage change conflates two distinct mechanisms: individual CPC changes and click volume mix shifts. Shapley has no way to separate them, so the attribution it produces reflects both entangled together. Attributing percentage CPC change via Shapley is effectively attributing the wrong thing.

# The proper solution

As we have been hinting in previous sections, the contribution of how and individual campaign contributes the global CPC surge depends on the combination of two factors:
- Its CPC change

- How its weight on the total number of clicks shifts

If we denote by $w_i = \text{clicks}_i / \text{total\_clicks}$ the clicks weight of campaign and we use superscripts we get the following nice decomposition:

$$\Delta{\text{CPC}}_{\text{global}} = \sum_i w_i^{\text{after}}\cdot\Delta{\text{CPC}_i} + \sum_i\Delta w_i\cdot\text{CPC}_i^{\text{before}}$$


In the sum, 
- The left term measures **rate effects**: how each individual CPC campaign has changed weighted by the current proportion of total clicks.

- The right term measures are **mix effects**: how the distribution of clicks accross the campaigns has shifted, changing how much weight each campaign's CPC carries in the aggregate.

In [ ]:
def plot_cpc_comparison(df: pd.DataFrame) -> None:
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df["cpc_before"],
            y=df["cpc_after"],
            mode="markers",
            marker_size=4,
            marker_color=accent_colors[0],
            text=df["product_id"],
            name="CPC before/after"
        )
    )

    fig.add_trace(
        go.Scatter(
            x=[0,80],
            y=[0,80],
            mode="lines",
            line=dict(color="red", dash="dash", width=1),
            name="Same CPC before/after"
        )
    )

    fig.update_layout(
        title="CPC comparison (before vs after)",
        width=600,
        height=600,
        xaxis_title="CPC before",
        yaxis_title="CPC after",
        yaxis_scaleanchor="x",
        yaxis_scaleratio=1,
        template="plotly_dark"
    )

    fig.show()

In [ ]:
plot_cpc_comparison(df)

In [ ]:
def plot_mix_rate_effect_comparison(df: pd.DataFrame) -> None:
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df["rate_effect"],
            y=df["mix_effect"],
            mode="markers",
            marker_size=4,
            marker_color=accent_colors[0],
            text=df["product_id"],
            name="Rate-mix effect"
        )
    )

    for i in range(-1,6):
        fig.add_trace(
            go.Scatter(
                x=[-0.02,0.02],
                y=[0.01 * i + 0.02, 0.01 * i - 0.02],
                mode="lines",
                line=dict(color="red", dash="dash", width=1),
                name=f"Total effect = {0.01 * i}"
            )
        )

    fig.update_layout(
        title="Rate vs Mix effect on CPC change",
        width=600,
        height=600,
        xaxis_title="Rate effect",
        yaxis_title="Mix effect",
        #yaxis_scaleanchor="x",
        #yaxis_scaleratio=1,
        xaxis_range=[-0.021, 0.021],
        showlegend=False,
        template="plotly_dark"
    )

    fig.show()

In [ ]:
# plot_cpc_comparison(df)
plot_mix_rate_effect_comparison(df)

**What to look for now**
The rate effect for product $i$ is $w_i^{\text{after}} \cdot \Delta\text{CPC}_i$. A product contributes strongly to the rate effect if:
- Its click weight is large and it gained a lot of CPC

The mix effect for product $i$ is $\Delta w_i \cdot \text{CPC}_i^{\text{before}}$​. A product contributes strongly to the mix effect if:
- It gained a lot of click share ($\Delta w_i$ is large and positive) and had a high baseline CPC



In [ ]:
def plot_mix_effect_factors(df: pd.DataFrame) -> None:
    ps_weight_delta = df["clicks_after"] / df["clicks_after"].sum() - df["clicks_before"] / df["clicks_before"].sum()
    
    fig = go.Figure()

    max_abs = np.abs(df["mix_effect"]).max()

    fig.add_trace(
        go.Scatter(
            x=ps_weight_delta,
            y=df["cpc_before"],
            mode="markers",
            marker=dict(
                size=5,
                color=df["mix_effect"],
                colorscale="RdBu_r",
                cmin=-max_abs,
                cmax=max_abs,
                colorbar=dict(title="Mix effect")
            ),
            text=df.apply(lambda row: f"{row['product_id']} mix effect = {round(row['mix_effect'], 6)}", axis=1)
        )
    )

    fig.update_layout(
        title="Mix effect factors",
        width=600,
        height=600,
        xaxis_title="Weight delta",
        yaxis_title="CPC before",
        showlegend=False,
        template="plotly_dark"
    )

    fig.show()

In [ ]:
def plot_rate_effect_factors(df: pd.DataFrame) -> None:
    ps_weight= df["clicks_after"] / df["clicks_after"].sum()
    
    fig = go.Figure()

    max_abs = np.abs(df["rate_effect"]).max()

    fig.add_trace(
        go.Scatter(
            x=df["cpc_diff"],
            y=ps_weight,
            mode="markers",
            marker=dict(
                size=5,
                color=df["rate_effect"],
                colorscale="RdBu_r",
                cmin=-max_abs,
                cmax=max_abs,
                colorbar=dict(title="Rate effect")
            ),
            text=df.apply(lambda row: f"{row['product_id']} rate effect = {round(row['rate_effect'], 6)}", axis=1)
        )
    )

    fig.update_layout(
        title="Rate effect factors",
        width=600,
        height=600,
        xaxis_title="CPC delta",
        yaxis_title="Clicks weight after",
        showlegend=False,
        template="plotly_dark"
    )

    fig.show()

In [ ]:
plot_rate_effect_factors(df)
plot_mix_effect_factors(df)

# Final comments

- Considering clicks from all campaigns as "the same product" (there are caveats like country or category)